# Exercise 1. Fine-Tuning for Multi-Label Classification 


## 1.1 Setup: Load Packages & Data

In [50]:
from pathlib import Path
from datasets import load_dataset, ClassLabel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

In [ ]:
path = Path.cwd()
data_path = path.parents[1] / "resources" / "data" / "hf" # path for huggingface datasets (so we don't have to redownload them every time)

In [ ]:
ds = load_dataset("SetFit/student-question-categories", split="train", cache_dir=data_path)

Repo card metadata block was not found. Setting CardData to empty.


Generating train split:   0%|          | 0/117519 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 117519
})


Print the dataset + an example of a text:

In [48]:
print(ds)

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 117519
})


In [49]:
# let's print an example
print(ds["text"][12])

Hydroponic is a subset of what type of culture?
A. Hydroculture
B. Solid medium culture
c. xeroculture
D. Tissue culture


Let's define the label column:

In [51]:
num_classes = 4
ds = ds.cast_column("label", ClassLabel(num_classes=num_classes))

Casting the dataset:   0%|          | 0/117519 [00:00<?, ? examples/s]

In [52]:
# split into train/val
ds = ds.train_test_split(train_size=2000,test_size=500, seed=42, stratify_by_column="label")
train_data = ds["train"]
val_data = ds["test"]

## 1.2 Loading the Model

In [58]:
# define where to load the model from
model_path = path.parents[1] / "resources" / "models" / "hf"

model_id = "distilbert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(
                                                            model_id, 
                                                            num_labels=num_classes, # pre-defined number of labels in our dataset
                                                            cache_dir=model_path, 
                                                           ) 
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=model_path)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Let's look more into our model by printing its parameters:

In [59]:
print(model.num_parameters())

65784580


:::{admonition} QUESTION
:class: red
`DistilBERT` has 65.M parameters. From what you might have heard about `Large Language Models` - do you know where this would range? Is this a lot?
:::

## 1.3 Tokenization
We can use BERT's trained tokenizer to represent the text in our dataset:

In [60]:
def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

:::{admonition} QUESTION
:class: red
Can you identify a way to make the function above better in terms of how it is defined and how it is described? Is it easily applicable to other datasets? Why/Why not?

<details>
  <summary>ANSWER</summary>
  I would consider to ...
  <ol>
    <li>Rename the function to <code>tokenize</code>, making its nane more informative to its purpose.</li>
    <li>Add more details in the docstring (and <a href="https://docs.python.org/3/library/typing.html">type hints</a>)about the expected input and output, instead of only writing <code>"""Tokenize input data"""</code>.</li>
    <li>Make the function more generalizable by adding a <code>text_col</code> parameter, allowing the user to specify a different column name (e.g., our text column being called <code>"generation"</code>)</li>
  </ol>
</details>
:::

We use the `.map` method to use the tokenize function on each row in our dataset!

In [ ]:
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_val = val_data.map(preprocess_function, batched=True)

## 1.4 Training Arguments and Parameters!
To `fine-tune` our BERT model, we'll use the `Trainer` class. This needs quite a lot of information. We'll break this down in this section!

```python
trainer = Trainer(
   model=model,
   args=training_args,            # how many examples per batch? how many times?
   tokenizer=tokenizer,             
   data_collator=data_collator,   # ensure equal length 
   compute_metrics=compute_metrics.
   train_dataset=tokenized_train, # data
   eval_dataset=tokenized_val,    # data
)
```

In [ ]:
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

Among other things, these arguments let's us define the speed at which BERT should learn (`learning_rate`), how many examples it should have in each batch of its training (`_batch_size`) and how many times it should train on all batches (`epochs`).

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    f1_metric = evaluate.load("f1")
    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"  # or "weighted"
    )["f1"]
    return {"f1": f1}

In [ ]:
# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

from transformers import TrainingArguments, Trainer

# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

/var/folders/gg/gk923hkx2w3bw72pk2shplydry9j0b/T/ipykernel_14085/3103918724.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_val,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

In [14]:
trainer.train()

/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=125, training_loss=0.3469104309082031, metrics={'train_runtime': 45.1655, 'train_samples_per_second': 44.282, 'train_steps_per_second': 2.768, 'total_flos': 136450426847616.0, 'train_loss': 0.3469104309082031, 'epoch': 1.0})

In [15]:
trainer.evaluate()

/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.3611033260822296,
 'eval_f1': 0.8846073358455832,
 'eval_runtime': 4.5413,
 'eval_samples_per_second': 110.1,
 'eval_steps_per_second': 7.046,
 'epoch': 1.0}

## Your Turn: Gather these in a Python Script!
:::{admonition} HANDS-ON
:class: red
To practice how to define scripts, I want you to take the snippets - from loading data all the way to training BERT and put them into a `main()` function!

You decide whether and if you want to define additional functions outside of `main()`!
:::